In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/grkmuls/train-with-features/train_with_features.csv
/kaggle/input/competitions/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv
/kaggle/input/competitions/learning-agency-lab-automated-essay-scoring-2/train.csv
/kaggle/input/competitions/learning-agency-lab-automated-essay-scoring-2/test.csv


In [ ]:
#versiyon kontrol 
import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "datasets", "accelerate"])

import pandas as pd
import numpy as np
import torch

print("GPU mevcut mu:", torch.cuda.is_available())
print("GPU adı:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Yok")

df_train = pd.read_csv("/kaggle/input/competitions/learning-agency-lab-automated-essay-scoring-2/train.csv")
df_test = pd.read_csv("/kaggle/input/competitions/learning-agency-lab-automated-essay-scoring-2/test.csv")

print(f"\nTrain: {df_train.shape}")
print(f"Test: {df_test.shape}")

GPU mevcut mu: True
GPU adı: Tesla T4

Train: (17307, 3)
Test: (3, 2)


In [ ]:
#tokenizer hücre2
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
import torch.nn as nn

# model
# deberta-v3-base → DeBERTa'nın üçüncü versiyonu, base boyutu
# Kaggle'da ücretsiz çalışır, large a göre daha hızlı
MODEL_NAME = "microsoft/deberta-v3-base"

# Sabit değerler
CONFIG = {
    "max_length": 512,      
    "batch_size": 4,
    "epochs": 2, 
    "lr": 2e-5,              
    "n_folds": 5,           
    "seed": 42              
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer yüklendi")

#  örnek 
sample = "This is a sample essay sentence."
tokens = tokenizer(sample, return_tensors="pt")
print(f"\nÖrnek token sayısı: {tokens['input_ids'].shape[1]}")
print(f"Token ID'leri: {tokens['input_ids'][0][:10]}...")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizer yüklendi

Örnek token sayısı: 7
Token ID'leri: tensor([ 329,  269,  266, 2783, 2415, 4378,  260])...


In [ ]:
class EssayDataset(Dataset):
    
    def __init__(self, texts, labels=None):
        self.texts = texts        # essay metinleri
        self.labels = labels      # skorlar (test setinde yok)
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        

        encoding = tokenizer(
            text,
            max_length=CONFIG["max_length"],
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        # gereksiz boyutu kaldır

        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze()
        }
        
        if self.labels is not None:
            # skor 1-6 arası ama model 0'dan başlayan index ister
            item["labels"] = torch.tensor(self.labels[idx] - 1, dtype=torch.long)
        
        return item

print("Dataset sınıfı tanımlandı")

# Test - örnek
sample_dataset = EssayDataset(
    texts=df_train["full_text"].tolist()[:3],
    labels=df_train["score"].tolist()[:3]
)

sample_loader = DataLoader(sample_dataset, batch_size=2)
batch = next(iter(sample_loader))  

print(f"input_ids boyutu: {batch['input_ids'].shape}")   # [2, 512] olmalı
print(f"labels: {batch['labels']}")                       

Dataset sınıfı tanımlandı
input_ids boyutu: torch.Size([2, 512])
labels: tensor([2, 2])


In [ ]:
#fonksiyon tanımlama
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# mixed precision eğitimi stabil
scaler = torch.cuda.amp.GradScaler()

def get_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=6,
        ignore_mismatched_sizes=True,
        torch_dtype=torch.float32  
    )
    return model.to(device)

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def eval_epoch(model, loader):
    model.eval()
    preds, trues = [], []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]
            
            with torch.cuda.amp.autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            
            predictions = outputs.logits.argmax(dim=1).cpu().numpy()
            preds.extend(predictions + 1)
            trues.extend(labels.numpy() + 1)
    
    qwk = cohen_kappa_score(trues, preds, weights="quadratic")
    return qwk

print("Fonksiyonlar tanımlandı")

Fonksiyonlar tanımlandı


/tmp/ipykernel_23/1354722075.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
import torch.nn as nn

# Aim 1 özellikleri
df_features = pd.read_csv("/kaggle/input/datasets/grkmuls/train-with-features/train_with_features.csv")

# Özellik sütunları
FEATURE_COLS = [
    "error_density",
    "ttr", 
    "mtld",
    "avg_sentence_length",
    "std_sentence_length",
    "dep_depth",
    "discourse_ratio"
]

print("Özellikler yüklendi:", df_features.shape)
print(df_features[FEATURE_COLS].describe().round(3))

# Özellikleri normalize etme
# farklı birimlerdeki özellikleri (hata oranı vs kelime sayısı) karşılaştırılabilir yapar
from sklearn.preprocessing import StandardScaler

# feature_scaler GradScaler yüzünden farklı isim
feature_scaler = StandardScaler()
features_scaled = feature_scaler.fit_transform(df_features[FEATURE_COLS])

print(f"\nNormalize edilmiş özellik matrisi: {features_scaled.shape}")

Özellikler yüklendi: (17307, 10)
       error_density        ttr       mtld  avg_sentence_length  \
count      17307.000  17307.000  17307.000            17307.000   
mean           0.054      0.514     72.787               21.698   
std            0.039      0.074     19.465               15.919   
min            0.000      0.055     26.283                6.406   
25%            0.026      0.463     59.094               16.250   
50%            0.044      0.512     70.359               19.214   
75%            0.071      0.564     83.814               23.176   
max            1.000      0.769    189.157              715.000   

       std_sentence_length  dep_depth  discourse_ratio  
count            17307.000  17307.000        17307.000  
mean                 9.576      2.662            0.002  
std                  7.123      0.710            0.003  
min                  0.000      0.500            0.000  
25%                  6.482      2.190            0.000  
50%                  

In [ ]:
#hybridmodel hücresi
class HybridModel(nn.Module):
    
    
    def __init__(self, n_linguistic_features=7):
        super().__init__()  # üst sınıfın __init__'ini çağır
        
        # DeBERTa backbone → metinden bağlamsal temsil öğrenir
        from transformers import AutoModel
        self.deberta = AutoModel.from_pretrained(MODEL_NAME, dtype=torch.float32)
        
        # DeBERTa'nın çıktı boyutu 768 (base model için sabit)
        deberta_hidden_size = 768
        
        # Linguistik özellikler için ön işleme katmanı
        # 7 özellik → 32 
        self.linguistic_layer = nn.Sequential(
            nn.Linear(n_linguistic_features, 32),  # Linear → tam bağlantılı katman
            nn.ReLU(),   # ReLU → negatif değerleri sıfırlar, non-linearity ekler
            nn.Dropout(0.1)  # Dropout → eğitimde rastgele nöronları kapatır, overfitting önler
        )
        
        # Birleştirme katmanı: DeBERTa (768) + linguistik (32) → 6 sınıf
        self.classifier = nn.Sequential(
            nn.Linear(deberta_hidden_size + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 6)  # 6 → skor sınıfı sayısı (1-6)
        )
    
    def forward(self, input_ids, attention_mask, linguistic_features):
        # forward → modelin ileri geçişi, her batch'te çağır
        
        # DeBERTa'dan [CLS] token temsilini al
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        
        # last_hidden_state[:, 0, :] → her örnek için [CLS] token vektörü
        # [CLS] tüm metnin özet temsili olarak kullanılcak
        cls_output = outputs.last_hidden_state[:, 0, :]  # [batch_size, 768]
        
        # Linguistik özellikleri işle
        ling_output = self.linguistic_layer(linguistic_features)  # [batch_size, 32]
        
        # İkisini birleştir
        # torch.cat → tensörleri belirtilen boyutta birleştir
        # dim=1 → sütun boyutunda birleştir: [batch_size, 768+32] = [batch_size, 800]
        combined = torch.cat([cls_output, ling_output], dim=1)
        
        # Sınıflandırma katmanından geçir
        logits = self.classifier(combined)  # [batch_size, 6]
        
        return logits

print("HybridModel tanımlandı")

# Test 
test_model = HybridModel().to(device)
print(f"Model parametreleri: {sum(p.numel() for p in test_model.parameters()):,}")


HybridModel tanımlandı


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Model parametreleri: 184,038,406


In [ ]:
# GradScaler tanımlaama, !StandardScaler
grad_scaler = torch.amp.GradScaler('cuda')

class HybridDataset(Dataset):
    def __init__(self, texts, linguistic_features, labels=None):
        self.texts = texts
        self.linguistic_features = torch.FloatTensor(linguistic_features)
        self.labels = labels
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=CONFIG["max_length"],
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "linguistic_features": self.linguistic_features[idx]
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx] - 1, dtype=torch.long)
        return item

def train_epoch_hybrid(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        ling_features = batch["linguistic_features"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            logits = model(input_ids, attention_mask, ling_features)
            loss = criterion(logits, labels)
        
        grad_scaler.scale(loss).backward()
        grad_scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        grad_scaler.step(optimizer)
        grad_scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def eval_epoch_hybrid(model, loader):
    model.eval()
    preds, trues = [], []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            ling_features = batch["linguistic_features"].to(device)
            labels = batch["labels"]
            
            with torch.amp.autocast('cuda'):
                logits = model(input_ids, attention_mask, ling_features)
            
            predictions = logits.argmax(dim=1).cpu().numpy()
            preds.extend(predictions + 1)
            trues.extend(labels.numpy() + 1)
    
    return cohen_kappa_score(trues, preds, weights="quadratic")

print("Hibrit model fonksiyonları tanımlandı")

Hibrit model fonksiyonları tanımlandı


In [9]:
torch.cuda.empty_cache()
CONFIG["batch_size"] = 4

In [ ]:

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

# essay sırası df_features ile aynı olmalı — essay_id üzerinden eşleştir
df_train_original = pd.read_csv("/kaggle/input/competitions/learning-agency-lab-automated-essay-scoring-2/train.csv")

# df_features zaten train.csv'den geldi, sıra aynı olmalı ama güvenli olmak için merge 
df_merged = df_train_original.merge(df_features[["essay_id"] + FEATURE_COLS], on="essay_id", how="left")

texts = df_merged["full_text"].tolist()
labels = df_merged["score"].tolist()
features = feature_scaler.transform(df_merged[FEATURE_COLS].values)
skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])

hybrid_fold_scores = []
baseline_fold_scores = [0.7552, 0.7606]  # baseline sonuçlar

MAX_FOLDS = 2
criterion = nn.CrossEntropyLoss()

for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
    if fold >= MAX_FOLDS:
        break
    
    print(f"\n{'='*50}")
    print(f"FOLD {fold+1}/{MAX_FOLDS}")
    print(f"{'='*50}")
    
    texts_arr = np.array(texts)
    labels_arr = np.array(labels)
    features_arr = features  

    train_dataset = HybridDataset(
        texts_arr[train_idx].tolist(),
        features_arr[train_idx],
        labels_arr[train_idx].tolist()
    )
    val_dataset = HybridDataset(
        texts_arr[val_idx].tolist(),
        features_arr[val_idx],
        labels_arr[val_idx].tolist()
    )
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)
    
    model = HybridModel().to(device)
    
    optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=0.01)
    total_steps = len(train_loader) * CONFIG["epochs"]
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    
    best_qwk = 0
    
    for epoch in range(CONFIG["epochs"]):
        print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
        
        train_loss = train_epoch_hybrid(model, train_loader, optimizer, scheduler, criterion)
        val_qwk = eval_epoch_hybrid(model, val_loader)
        
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val QWK:    {val_qwk:.4f}  (baseline: {baseline_fold_scores[fold]:.4f})")
        
        if val_qwk > best_qwk:
            best_qwk = val_qwk
            torch.save(model.state_dict(), f"/kaggle/working/hybrid_model_fold{fold+1}.pt")
            print(f"  ✓ Model kaydedildi (best QWK: {best_qwk:.4f})")
    
    hybrid_fold_scores.append(best_qwk)
    diff = best_qwk - baseline_fold_scores[fold]
    print(f"\nFold {fold+1} tamamlandı.")
    print(f"  Hibrit QWK:   {best_qwk:.4f}")
    print(f"  Baseline QWK: {baseline_fold_scores[fold]:.4f}")
    print(f"  Fark:         {diff:+.4f}")

print(f"\n{'='*50}")
print(f"Hibrit model ortalama QWK: {np.mean(hybrid_fold_scores):.4f}")
print(f"Baseline ortalama QWK:     {np.mean(baseline_fold_scores):.4f}")
print(f"Toplam fark:               {np.mean(hybrid_fold_scores) - np.mean(baseline_fold_scores):+.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



FOLD 1/2


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/2


/tmp/ipykernel_23/3603644330.py:51: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  Train Loss: 1.1414
  Val QWK:    0.7250  (baseline: 0.7552)
  ✓ Model kaydedildi (best QWK: 0.7250)

Epoch 2/2
  Train Loss: 0.8733
  Val QWK:    0.7496  (baseline: 0.7552)
  ✓ Model kaydedildi (best QWK: 0.7496)

Fold 1 tamamlandı.
  Hibrit QWK:   0.7496
  Baseline QWK: 0.7552
  Fark:         -0.0056

FOLD 2/2


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/2
  Train Loss: 1.1262
  Val QWK:    0.7436  (baseline: 0.7606)
  ✓ Model kaydedildi (best QWK: 0.7436)

Epoch 2/2
  Train Loss: 0.8620
  Val QWK:    0.7576  (baseline: 0.7606)
  ✓ Model kaydedildi (best QWK: 0.7576)

Fold 2 tamamlandı.
  Hibrit QWK:   0.7576
  Baseline QWK: 0.7606
  Fark:         -0.0030

Hibrit model ortalama QWK: 0.7536
Baseline ortalama QWK:     0.7579
Toplam fark:               -0.0043


In [ ]:
#ablasyon
import numpy as np

# Özellik grupları — her grubu sırayla sıfırlayıp etkisini ölç
FEATURE_GROUPS = {
    "Hata Yoğunluğu":        ["error_density"],
    "Sözcüksel Çeşitlilik":  ["ttr", "mtld"],
    "Sözdizimsel Karmaşıklık": ["avg_sentence_length", "std_sentence_length", "dep_depth"],
    "Discourse Bağlacı":     ["discourse_ratio"]
}

skf_temp = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])
folds_temp = list(skf_temp.split(texts, labels))
_, val_idx = folds_temp[0]  # fold 1'in val seti

texts_arr = np.array(texts)
labels_arr = np.array(labels)

# Kaydedilen hibrit modeli 
ablation_model = HybridModel().to(device)
ablation_model.load_state_dict(torch.load("/kaggle/working/hybrid_model_fold1.pt"))

# Önce tüm özelliklerle baseline QWK
features_val = feature_scaler.transform(df_merged[FEATURE_COLS].iloc[val_idx].values)

val_dataset_full = HybridDataset(
    texts_arr[val_idx].tolist(),
    features_val,
    labels_arr[val_idx].tolist()
)
val_loader_full = DataLoader(val_dataset_full, batch_size=CONFIG["batch_size"], shuffle=False)
full_qwk = eval_epoch_hybrid(ablation_model, val_loader_full)
print(f"Tüm özelliklerle QWK: {full_qwk:.4f}")

# Her grubu sırayla sıfırla
print(f"\n{'Özellik Grubu':<30} {'QWK':>8} {'Fark':>8}")
print("-" * 50)

for group_name, group_cols in FEATURE_GROUPS.items():
    # Özelliklerin kopyasını al, o grubun sütunlarını sıfırla
    features_ablated = df_merged[FEATURE_COLS].copy()
    features_ablated[group_cols] = 0.0  # o grubu sıfırla
    
    features_ablated_scaled = feature_scaler.transform(features_ablated.iloc[val_idx].values)
    
    val_dataset_ablated = HybridDataset(
        texts_arr[val_idx].tolist(),
        features_ablated_scaled,
        labels_arr[val_idx].tolist()
    )
    val_loader_ablated = DataLoader(val_dataset_ablated, batch_size=CONFIG["batch_size"], shuffle=False)
    
    ablated_qwk = eval_epoch_hybrid(ablation_model, val_loader_ablated)
    diff = ablated_qwk - full_qwk
    
    print(f"{group_name:<30} {ablated_qwk:>8.4f} {diff:>+8.4f}")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/sklearn/utils

Tüm özelliklerle QWK: 0.7496

Özellik Grubu                       QWK     Fark
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Hata Yoğunluğu                   0.7493  -0.0004


...

Sözcüksel Çeşitlilik             0.7473  -0.0023


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Sözdizimsel Karmaşıklık          0.7498  +0.0002


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Discourse Bağlacı                0.7498  +0.0002


In [ ]:
from sklearn.metrics import f1_score, classification_report

# Val seti tahminleri
ablation_model.eval()
all_preds, all_trues = [], []

with torch.no_grad():
    for batch in val_loader_full:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        ling_features = batch["linguistic_features"].to(device)
        labels = batch["labels"]
        
        with torch.amp.autocast('cuda'):
            logits = ablation_model(input_ids, attention_mask, ling_features)
        
        predictions = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(predictions + 1)
        all_trues.extend(labels.numpy() + 1)

# Her skor sınıfı için F1
print("Sınıf bazlı F1 skorları:")
print(classification_report(
    all_trues, all_preds,
    labels=[1,2,3,4,5,6],
    target_names=["Skor 1","Skor 2","Skor 3","Skor 4","Skor 5","Skor 6"]
))

Sınıf bazlı F1 skorları:
              precision    recall  f1-score   support

      Skor 1       0.76      0.31      0.45       251
      Skor 2       0.72      0.54      0.62       944
      Skor 3       0.61      0.66      0.63      1256
      Skor 4       0.54      0.74      0.63       786
      Skor 5       0.39      0.40      0.39       194
      Skor 6       0.00      0.00      0.00        31

    accuracy                           0.60      3462
   macro avg       0.50      0.44      0.45      3462
weighted avg       0.62      0.60      0.59      3462



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#Bu ESKİ eğitimi
import numpy as np
from sklearn.model_selection import StratifiedKFold

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

texts = df_train["full_text"].tolist()
labels = df_train["score"].tolist()


skf = StratifiedKFold(n_splits=CONFIG["n_folds"], shuffle=True, random_state=CONFIG["seed"])

fold_scores = []

# performans kısıtlamasından, 2 fold baseline 5 yerine
MAX_FOLDS = 2

for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
    if fold >= MAX_FOLDS:
        break
    
    print(f"\n{'='*50}")
    print(f"FOLD {fold+1}/{MAX_FOLDS}")
    print(f"{'='*50}")
    
    texts_arr = np.array(texts)
    labels_arr = np.array(labels)
    
    train_dataset = EssayDataset(texts_arr[train_idx].tolist(), labels_arr[train_idx].tolist())
    val_dataset = EssayDataset(texts_arr[val_idx].tolist(), labels_arr[val_idx].tolist())
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)
    
    model = get_model()
    
    optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=0.01)
    
    # toplam adım sayısı = batch sayısı x epoch sayısı
    total_steps = len(train_loader) * CONFIG["epochs"]
    warmup_steps = int(total_steps * 0.1)  # ilk %10'da lr yavaş artar
    
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    best_qwk = 0
    
    for epoch in range(CONFIG["epochs"]):
        print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")
        
        train_loss = train_epoch(model, train_loader, optimizer, scheduler)
        val_qwk = eval_epoch(model, val_loader)
        
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val QWK:    {val_qwk:.4f}")
        
        if val_qwk > best_qwk:
            best_qwk = val_qwk
            # En iyi modeli kaydet
            torch.save(model.state_dict(), f"/kaggle/working/model_fold{fold+1}.pt")
            print(f"  ✓ Model kaydedildi (best QWK: {best_qwk:.4f})")
    
    fold_scores.append(best_qwk)
    print(f"\nFold {fold+1} tamamlandı. Best QWK: {best_qwk:.4f}")

print(f"\n{'='*50}")
print(f"Baseline sonuçları:")
print(f"Fold skorları: {[f'{s:.4f}' for s in fold_scores]}")
print(f"Ortalama QWK: {np.mean(fold_scores):.4f}")


FOLD 1/2


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight      


Epoch 1/2


...


Epoch 1/2


/tmp/ipykernel_23/1354722075.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/1354722075.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Train Loss: 1.1214
  Val QWK:    0.6945
  ✓ Model kaydedildi (best QWK: 0.6945)

Epoch 2/2


/tmp/ipykernel_23/1354722075.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_23/1354722075.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


  Train Loss: 0.8597
  Val QWK:    0.7640
  ✓ Model kaydedildi (best QWK: 0.7640)

Fold 2 tamamlandı. Best QWK: 0.7640

Baseline sonuçları:
Fold skorları: ['0.7580', '0.7640']
Ortalama QWK: 0.7610


In [14]:
# GPU belleğini temizle
import gc
torch.cuda.empty_cache()
gc.collect()
print(f"GPU belleği: {torch.cuda.memory_allocated()/1024**3:.2f} GB kullanımda")

GPU belleği: 4.17 GB kullanımda


In [15]:
from IPython.display import FileLink
FileLink(r'hybrid_model_fold1.pt')


/kaggle/working/hybrid_model_fold1.pt